<a href="https://colab.research.google.com/github/EugeneOny1/-civil-infrastructure-monitoring---CS-PROJECT-II/blob/feature-branch/05_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook 05 — SSD-MobileNetV2 Model Training

This notebook fine-tunes SSD-MobileNetV2 using the TensorFlow Object Detection API.

**Run this notebook in a Colab GPU runtime.**

Recommended:
`Runtime → Change runtime type → T4 GPU` when available.

Training artifacts should remain outside Git unless intentionally documented

In [ ]:
# Environment check
import os
import sys
from pathlib import Path
import shutil
import subprocess

import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print("WARNING: No GPU detected. Do not begin a long training run on CPU.")

## 1. Project Paths

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/CS PROJECT II")
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "Processed Data"

TFRECORD_DIR = PROCESSED_DIR / "tfrecords"

LABEL_MAP_PATH = PROJECT_DIR / "configs" / "label_map.pbtxt"
BASE_PIPELINE_CONFIG = PROJECT_DIR / "configs" / "pipeline.config"

TRAIN_RECORD = TFRECORD_DIR / "train.record"
VAL_RECORD = TFRECORD_DIR / "validation.record"
TEST_RECORD = TFRECORD_DIR / "test.record"

TRAINING_DIR = PROJECT_DIR / "training_checkpoints"
EXPORT_DIR = PROJECT_DIR / "exported_model"

print("Base pipeline:", BASE_PIPELINE_CONFIG)
print("Train record:", TRAIN_RECORD)
print("Validation record:", VAL_RECORD)

## 2. Install TensorFlow Object Detection API

Use a fresh Colab runtime if dependency installation produces incompatible TensorFlow/protobuf errors.

In [ ]:
# TensorFlow Models repository
if not Path("/content/models").exists():
    !git clone --depth 1 https://github.com/tensorflow/models.git /content/models

%cd /content/models/research

!protoc object_detection/protos/*.proto --python_out=.

!cp object_detection/packages/tf2/setup.py /content/models/research/setup.py
!pip install -q .

%cd /content

In [ ]:
# Verify Object Detection API import.
from object_detection.utils import config_util
from object_detection.protos import pipeline_pb2

print("TensorFlow Object Detection API import: OK")

## 3. Download the Pretrained SSD-MobileNetV2 Checkpoint

The checkpoint provides transfer-learning weights. The final model will be fine-tuned for the project's three classes.

In [ ]:
PRETRAINED_NAME = "ssd_mobilenet_v2_320x320_coco17_tpu-8"
PRETRAINED_URL = (
    "http://download.tensorflow.org/models/object_detection/tf2/20200711/"
    "ssd_mobilenet_v2_320x320_coco17_tpu-8.tar.gz"
)

PRETRAINED_ROOT = Path("/content/pretrained")
PRETRAINED_ROOT.mkdir(exist_ok=True)

archive = PRETRAINED_ROOT / f"{PRETRAINED_NAME}.tar.gz"
model_dir = PRETRAINED_ROOT / PRETRAINED_NAME

if not model_dir.exists():
    !wget -q -O "{archive}" "{PRETRAINED_URL}"
    !tar -xzf "{archive}" -C "{PRETRAINED_ROOT}"

print("Checkpoint directory:", model_dir)

## 4. Prepare a Runtime Pipeline Configuration

The repository `configs/pipeline.config` is treated as a project configuration reference. This notebook creates a runtime copy rather than silently overwriting the repository file.

**If the existing pipeline config is a placeholder or uses a different model architecture, inspect it before training.**

In [ ]:
RUNTIME_CONFIG = PROJECT_DIR / "configs" / "pipeline_runtime.config"
RUNTIME_CONFIG.parent.mkdir(exist_ok=True)

if not BASE_PIPELINE_CONFIG.exists():
    raise FileNotFoundError(
        f"Missing {BASE_PIPELINE_CONFIG}. "
        "Application scaffolding may have created a placeholder; create/replace it with the SSD-MobileNetV2 pipeline config before training."
    )

shutil.copy2(BASE_PIPELINE_CONFIG, RUNTIME_CONFIG)

print("Runtime config:", RUNTIME_CONFIG)

## 5. Patch the Pipeline Configuration

This uses the TensorFlow Object Detection API protobuf rather than fragile string replacement.

The model is configured for:
- 3 classes
- batch size 8 by default
- pretrained SSD-MobileNetV2 checkpoint
- train/validation TFRecords
- project label map

In [ ]:
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2

pipeline_config = pipeline_pb2.TrainEvalPipelineConfig()

with open(RUNTIME_CONFIG, "r", encoding="utf-8") as f:
    text_format.Merge(f.read(), pipeline_config)

# Model
pipeline_config.model.ssd.num_classes = 3

# Training
pipeline_config.train_config.batch_size = 8
pipeline_config.train_config.fine_tune_checkpoint = str(
    model_dir / "checkpoint" / "ckpt-0"
)
pipeline_config.train_config.fine_tune_checkpoint_type = "detection"

# Train input
pipeline_config.train_input_reader.label_map_path = str(LABEL_MAP_PATH)
pipeline_config.train_input_reader.tf_record_input_reader.input_path[:] = [
    str(TRAIN_RECORD)
]

# Eval input
pipeline_config.eval_input_reader[0].label_map_path = str(LABEL_MAP_PATH)
pipeline_config.eval_input_reader[0].tf_record_input_reader.input_path[:] = [
    str(VAL_RECORD)
]

# Write runtime config
with open(RUNTIME_CONFIG, "w", encoding="utf-8") as f:
    f.write(text_format.MessageToString(pipeline_config))

print("Runtime pipeline configured:", RUNTIME_CONFIG)

## 6. Training

Start with a manageable number of steps and inspect the loss before committing to a long run.

**Do not assume a specific number of steps is sufficient for the final model.** The final number should be based on validation behaviour and available training time.

In [ ]:
TRAIN_STEPS = 10000  # Initial run. Increase after inspecting convergence.

MODEL_MAIN = Path("/content/models/research/object_detection/model_main_tf2.py")

TRAINING_DIR.mkdir(parents=True, exist_ok=True)

command = [
    "python",
    str(MODEL_MAIN),
    f"--pipeline_config_path={RUNTIME_CONFIG}",
    f"--model_dir={TRAINING_DIR}",
    f"--alsologtostderr",
]

print("Training command:")
print(" ".join(command))

# Uncomment to start training:
# subprocess.run(command, check=True)

## 7. TensorBoard

Run TensorBoard to inspect training loss and evaluation metrics.

```python
%load_ext tensorboard
%tensorboard --logdir "/content/drive/MyDrive/CS PROJECT II/training_checkpoints"
```

Look for:
- total loss
- classification loss
- localization loss
- learning behaviour
- evaluation metrics when evaluation is running

In [ ]:
# List checkpoint artifacts after training.
checkpoints = sorted(TRAINING_DIR.glob("ckpt-*"))
print("Checkpoint artifacts:")
for p in checkpoints[-10:]:
    print(p)

### Application integration checkpoint

Notebook 05 does not need the Flask application.

The application will only need the **exported inference model** produced by Notebook 07.

Before deployment, the application team must confirm:
1. which exported format it will consume (`SavedModel` and/or `.tflite`);
2. expected model input size;
3. class ID ordering;
4. output tensor names/shapes if using TFLite;
5. model artifact path expected by `backend/app/services/inference.py`.

Do not place large model weights into Git unless deliberately required.